# Grokking as a toy task: running-average trajectories and PCA

This notebook trains a small quadratic network on **modular addition mod 23** using genuine noisy minibatch SGD on **`cuda:7`**. It is deliberately trained long after memorization so that delayed held-out generalization (grokking) is visible.

The model receives $(a,b)$ and predicts $(a+b)\bmod 23$. Train/test assignment is made by unordered pair orbit, so $(a,b)$ and $(b,a)$ always belong to the same split; swapped-pair leakage is impossible.

There are 12,000 optimizer updates and exactly 1,000 saved post-update endpoints

$$t_i = 12(i+1),\qquad i=0,\ldots,999.$$

Let $w^{(t)}$ be the flattened parameters after actual optimizer update $t$. At endpoint $t_i$, the saved raw path is $w_i=w^{(t_i)}$, while

$$\bar w_i = \frac{1}{\min(50,t_i)}\sum_{t=\max(1,t_i-49)}^{t_i} w^{(t)}.$$

Thus every average after update 50 contains the preceding **50 actual SGD iterates**, not 50 spaced checkpoints. PCA is fit only to the 1,000 averaged vectors. Both averaged and raw paths use that same PCA mean and basis; the top-2 plot uses the first two directions from the top-3 fit.

The modular-addition setup follows the small algorithmic tasks used in [Power et al. (2022)](https://arxiv.org/abs/2201.02177), while the quadratic architecture is based on the interpretable toy model in [Gromov (2023)](https://openreview.net/forum?id=0ZUKLCxwBo).


In [ ]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

SEED = 2026
MODULUS = 23
HIDDEN_WIDTH = 128
BATCH_SIZE = 32
TOTAL_UPDATES = 12_000
N_CHECKPOINTS = 1_000
CHECKPOINT_STRIDE = TOTAL_UPDATES // N_CHECKPOINTS
AVERAGE_WINDOW = 50
LEARNING_RATE = 3.0
WEIGHT_DECAY = 1e-4
SUSTAINED_CHECKPOINTS = 5
OUT_DIR = Path("diff_plots_2")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert CHECKPOINT_STRIDE == 12
assert N_CHECKPOINTS * CHECKPOINT_STRIDE == TOTAL_UPDATES

try:
    gpu_devices = jax.devices("gpu")
except RuntimeError as exc:
    raise RuntimeError("This notebook requires a CUDA-enabled JAX installation.") from exc
if len(gpu_devices) <= 7:
    raise RuntimeError(
        f"cuda:7 was requested, but only {len(gpu_devices)} GPU device(s) are visible."
    )
DEVICE = gpu_devices[7]
if DEVICE.id != 7:
    raise RuntimeError(f"Expected physical cuda:7, selected {DEVICE}.")

print(f"JAX {jax.__version__}")
print(f"Using {DEVICE} ({DEVICE.device_kind})")
print(f"Plots will be written to {OUT_DIR.resolve()}")


## Transpose-safe modular-addition split

There are $23^2=529$ ordered input pairs. We randomly assign half of the 276 unordered-pair orbits to training and the rest to testing. This produces a nearly even split while keeping both orientations of every off-diagonal pair together. Every output residue occurs in both splits.


In [ ]:
# Build the split on the host, then place all training arrays on cuda:7.
rng = np.random.default_rng(SEED)
upper_a, upper_b = np.triu_indices(MODULUS)
group_order = rng.permutation(upper_a.size)
n_train_groups = upper_a.size // 2
selected_groups = group_order[:n_train_groups]

train_matrix = np.zeros((MODULUS, MODULUS), dtype=bool)
train_matrix[upper_a[selected_groups], upper_b[selected_groups]] = True
train_matrix |= train_matrix.T
assert np.array_equal(train_matrix, train_matrix.T)

operand_a, operand_b = np.meshgrid(
    np.arange(MODULUS), np.arange(MODULUS), indexing="ij"
)
all_pairs_np = np.stack([operand_a.ravel(), operand_b.ravel()], axis=1)
all_targets_np = (all_pairs_np[:, 0] + all_pairs_np[:, 1]) % MODULUS
train_mask_np = train_matrix.ravel()

train_pairs_np = all_pairs_np[train_mask_np]
train_targets_np = all_targets_np[train_mask_np]
test_pairs_np = all_pairs_np[~train_mask_np]
test_targets_np = all_targets_np[~train_mask_np]

for residue in range(MODULUS):
    assert np.any(train_targets_np == residue)
    assert np.any(test_targets_np == residue)


def init_params(key):
    key_in, key_out = jax.random.split(key)
    input_limit = 1.0 / np.sqrt(2 * MODULUS)
    output_limit = 1.0 / np.sqrt(HIDDEN_WIDTH)
    w_input = jax.random.uniform(
        key_in,
        (2 * MODULUS, HIDDEN_WIDTH),
        minval=-input_limit,
        maxval=input_limit,
        dtype=jnp.float32,
    )
    w_output = jax.random.uniform(
        key_out,
        (HIDDEN_WIDTH, MODULUS),
        minval=-output_limit,
        maxval=output_limit,
        dtype=jnp.float32,
    )
    return (w_input, w_output)


def model_logits(params, pairs):
    w_input, w_output = params
    left = jax.nn.one_hot(pairs[:, 0], MODULUS) @ w_input[:MODULUS]
    right = jax.nn.one_hot(pairs[:, 1], MODULUS) @ w_input[MODULUS:]
    hidden = (left + right) ** 2
    return hidden @ w_output


def mse_loss(params, pairs, targets):
    predictions = model_logits(params, pairs)
    one_hot_targets = jax.nn.one_hot(targets, MODULUS)
    # This exact mean over batch × classes is part of the tuned SGD setup.
    return jnp.mean((predictions - one_hot_targets) ** 2)


def flatten_params(params):
    return jnp.concatenate([parameter.reshape(-1) for parameter in params])


with jax.default_device(DEVICE):
    train_pairs = jnp.asarray(train_pairs_np, dtype=jnp.int32)
    train_targets = jnp.asarray(train_targets_np, dtype=jnp.int32)
    test_pairs = jnp.asarray(test_pairs_np, dtype=jnp.int32)
    test_targets = jnp.asarray(test_targets_np, dtype=jnp.int32)

    root_key = jax.random.key(SEED)
    init_key, batch_key = jax.random.split(root_key)
    initial_params = init_params(init_key)
    batch_indices = jax.random.randint(
        batch_key,
        shape=(N_CHECKPOINTS, CHECKPOINT_STRIDE, BATCH_SIZE),
        minval=0,
        maxval=train_pairs.shape[0],
        dtype=jnp.int32,
    )

parameter_count = int(flatten_params(initial_params).size)
assert parameter_count == 8_832
assert all(parameter.devices() == {DEVICE} for parameter in initial_params)
assert train_pairs.devices() == {DEVICE}
assert batch_indices.devices() == {DEVICE}

print(f"Train pairs: {train_pairs.shape[0]}; held-out pairs: {test_pairs.shape[0]}")
print(f"Quadratic network parameters: {parameter_count:,}")
print(f"Chance accuracy: {100.0 / MODULUS:.2f}%")


## 12,000 updates of true minibatch SGD

The update is

$$\theta \leftarrow \theta - \eta\left(\nabla_\theta L_{\mathrm{batch}} + \lambda\theta\right),$$

with no momentum or adaptive state. This is plain SGD with coupled L2 regularization $\frac{\lambda}{2}\lVert\theta\rVert_2^2$. A 50-vector GPU ring buffer is updated after **every** optimizer step. Only the 1,000 scheduled raw vectors and exact rolling averages are retained.


In [ ]:
@jax.jit(device=DEVICE)
def run_sgd_and_record(
    params,
    all_train_pairs,
    all_train_targets,
    all_test_pairs,
    all_test_targets,
    all_batch_indices,
):
    n_params = flatten_params(params).size
    history_buffer = jnp.zeros((AVERAGE_WINDOW, n_params), dtype=jnp.float32)
    rolling_sum = jnp.zeros((n_params,), dtype=jnp.float32)
    update_number = jnp.asarray(0, dtype=jnp.int32)

    def checkpoint_block(carry, block_indices):
        def update_once(inner_carry, indices):
            current_params, buffer, current_sum, current_update = inner_carry
            batch_loss, gradients = jax.value_and_grad(mse_loss)(
                current_params,
                all_train_pairs[indices],
                all_train_targets[indices],
            )
            next_params = jax.tree_util.tree_map(
                lambda parameter, gradient: parameter
                - LEARNING_RATE * (gradient + WEIGHT_DECAY * parameter),
                current_params,
                gradients,
            )

            current_update = current_update + 1
            flat_params = flatten_params(next_params)
            slot = (current_update - 1) % AVERAGE_WINDOW
            outgoing = buffer[slot]
            buffer = buffer.at[slot].set(flat_params)
            outgoing_if_full = jnp.where(
                current_update > AVERAGE_WINDOW,
                outgoing,
                jnp.zeros_like(outgoing),
            )
            current_sum = current_sum + flat_params - outgoing_if_full
            return (
                next_params,
                buffer,
                current_sum,
                current_update,
            ), batch_loss

        carry, block_losses = jax.lax.scan(update_once, carry, block_indices)
        current_params, buffer, current_sum, current_update = carry
        raw_vector = flatten_params(current_params)
        average_count = jnp.minimum(current_update, AVERAGE_WINDOW).astype(jnp.float32)
        average_vector = current_sum / average_count

        train_loss = mse_loss(current_params, all_train_pairs, all_train_targets)
        test_loss = mse_loss(current_params, all_test_pairs, all_test_targets)
        train_accuracy = jnp.mean(
            jnp.argmax(model_logits(current_params, all_train_pairs), axis=1)
            == all_train_targets
        )
        test_accuracy = jnp.mean(
            jnp.argmax(model_logits(current_params, all_test_pairs), axis=1)
            == all_test_targets
        )
        parameter_norm = jnp.linalg.norm(raw_vector)

        checkpoint_values = (
            raw_vector,
            average_vector,
            train_loss,
            test_loss,
            train_accuracy,
            test_accuracy,
            parameter_norm,
            block_losses[-1],
        )
        return carry, checkpoint_values

    initial_carry = (params, history_buffer, rolling_sum, update_number)
    return jax.lax.scan(checkpoint_block, initial_carry, all_batch_indices)


final_carry, recorded = run_sgd_and_record(
    initial_params,
    train_pairs,
    train_targets,
    test_pairs,
    test_targets,
    batch_indices,
)
(
    w_sgd,
    w_avg,
    train_losses,
    test_losses,
    train_accuracies,
    test_accuracies,
    parameter_norms,
    minibatch_losses,
) = recorded
w_avg.block_until_ready()

final_params, final_buffer, final_rolling_sum, final_update = final_carry
checkpoint_steps = CHECKPOINT_STRIDE * np.arange(1, N_CHECKPOINTS + 1)

assert int(final_update) == TOTAL_UPDATES
assert checkpoint_steps[0] == CHECKPOINT_STRIDE
assert checkpoint_steps[-1] == TOTAL_UPDATES
assert w_sgd.shape == w_avg.shape == (N_CHECKPOINTS, parameter_count)
assert w_sgd.devices() == {DEVICE}
assert w_avg.devices() == {DEVICE}
assert train_accuracies.devices() == {DEVICE}

print(f"Completed {int(final_update):,} SGD updates on {w_sgd.devices()}")
print(f"Recorded raw/average paths: {w_sgd.shape}")
print(
    "Each checkpoint average contains the preceding 50 actual updates "
    "once update 50 is reached."
)


## Verify delayed generalization

Metrics are evaluated on the complete train and held-out splits every 12 optimizer updates. To avoid declaring a transition from a one-checkpoint fluctuation, an event must remain above its threshold for five consecutive checkpoints (60 updates).


In [ ]:
(
    train_losses_np,
    test_losses_np,
    train_accuracies_np,
    test_accuracies_np,
    parameter_norms_np,
) = [
    np.asarray(jax.device_get(array))
    for array in (
        train_losses,
        test_losses,
        train_accuracies,
        test_accuracies,
        parameter_norms,
    )
]


def first_sustained_index(values, threshold, length=SUSTAINED_CHECKPOINTS):
    hits = (values >= threshold).astype(np.int32)
    run_lengths = np.convolve(hits, np.ones(length, dtype=np.int32), mode="valid")
    candidates = np.flatnonzero(run_lengths == length)
    return int(candidates[0]) if candidates.size else None


memorization_index = first_sustained_index(train_accuracies_np, 0.99)
grok_90_index = first_sustained_index(test_accuracies_np, 0.90)
grok_95_index = first_sustained_index(test_accuracies_np, 0.95)

if memorization_index is None:
    raise RuntimeError("Training accuracy never sustained 99%; this run did not memorize.")
if grok_90_index is None:
    raise RuntimeError("Held-out accuracy never sustained 90%; this run did not grok.")

memorization_step = int(checkpoint_steps[memorization_index])
grok_90_step = int(checkpoint_steps[grok_90_index])
grok_95_step = (
    int(checkpoint_steps[grok_95_index]) if grok_95_index is not None else None
)
grokking_lag = grok_90_step - memorization_step

assert grok_90_step > memorization_step
assert test_accuracies_np[memorization_index] < 0.20
assert test_accuracies_np[-1] >= 0.90

print(f"Sustained train accuracy ≥99%: update {memorization_step:,}")
print(f"Sustained test accuracy  ≥90%: update {grok_90_step:,}")
if grok_95_step is not None:
    print(f"Sustained test accuracy  ≥95%: update {grok_95_step:,}")
print(f"Memorization-to-grokking lag:  {grokking_lag:,} updates")
print(f"Final train accuracy: {100.0 * train_accuracies_np[-1]:.2f}%")
print(f"Final test accuracy:  {100.0 * test_accuracies_np[-1]:.2f}%")


## PCA of the 1,000 exact trailing averages

Because there are many more parameters (8,832) than checkpoints (1,000), PCA is computed exactly through the $1000\times1000$ sample Gram matrix. If $V_3$ contains the top three right singular vectors of the centered averaged path, the projections are

$$Z_{\mathrm{avg}}=(W_{\mathrm{avg}}-\mu)V_3,\qquad
Z_{\mathrm{sgd}}=(W_{\mathrm{sgd}}-\mu)V_3.$$

The raw SGD path is not centered independently.


In [ ]:
with jax.default_device(DEVICE):
    pca_mean = jnp.mean(w_avg, axis=0)
    centered_avg = w_avg - pca_mean

    # Exact dual PCA: eigenvectors of X X^T are left singular vectors of X.
    gram_matrix = centered_avg @ centered_avg.T
    eigenvalues, left_vectors = jnp.linalg.eigh(gram_matrix)
    descending_eigenvalues = jnp.flip(eigenvalues)
    top_eigenvalues = descending_eigenvalues[:3]
    top_left_vectors = jnp.flip(left_vectors[:, -3:], axis=1)
    top_singular_values = jnp.sqrt(jnp.maximum(top_eigenvalues, 1e-12))
    components_3 = centered_avg.T @ (
        top_left_vectors / top_singular_values[None, :]
    )
    components_3 = components_3 / jnp.linalg.norm(components_3, axis=0, keepdims=True)

    avg_3d = centered_avg @ components_3
    sgd_3d = (w_sgd - pca_mean) @ components_3

    components_2 = components_3[:, :2]
    avg_2d = centered_avg @ components_2
    sgd_2d = (w_sgd - pca_mean) @ components_2

    total_centered_energy = jnp.sum(centered_avg**2)
    explained_ratio = jnp.maximum(descending_eigenvalues, 0.0) / total_centered_energy

avg_3d.block_until_ready()

assert gram_matrix.shape == (N_CHECKPOINTS, N_CHECKPOINTS)
assert components_3.shape == (parameter_count, 3)
assert avg_3d.shape == sgd_3d.shape == (N_CHECKPOINTS, 3)
assert avg_3d.devices() == {DEVICE}
assert sgd_3d.devices() == {DEVICE}
np.testing.assert_allclose(
    np.asarray(jax.device_get(avg_2d)),
    np.asarray(jax.device_get(avg_3d[:, :2])),
    rtol=1e-5,
    atol=1e-6,
)
np.testing.assert_allclose(
    np.asarray(jax.device_get(components_3.T @ components_3)),
    np.eye(3),
    rtol=2e-3,
    atol=2e-3,
)

top3_ratio = np.asarray(jax.device_get(explained_ratio[:3]))
print("Top-3 explained-variance ratios:", np.round(top3_ratio, 6))
print(f"Top-3 cumulative explained variance: {100.0 * top3_ratio.sum():.2f}%")


## Save grokking and trajectory plots

The diagnostic curves establish the memorization/generalization delay. PCA plots overlay the noisy raw SGD checkpoints with their exact 50-update averages and mark memorization, grokking, and the final point.


In [ ]:
(
    sgd_3d_np,
    avg_3d_np,
    sgd_2d_np,
    avg_2d_np,
    explained_ratio_np,
) = [
    np.asarray(jax.device_get(array))
    for array in (sgd_3d, avg_3d, sgd_2d, avg_2d, explained_ratio)
]

chance_accuracy = 1.0 / MODULUS


def add_event_lines(ax):
    ax.axvline(
        memorization_step,
        color="#B279A2",
        linestyle="--",
        linewidth=1.5,
        label=f"Train ≥99% (step {memorization_step:,})",
    )
    ax.axvline(
        grok_90_step,
        color="#54A24B",
        linestyle="--",
        linewidth=1.5,
        label=f"Test ≥90% (step {grok_90_step:,})",
    )
    if grok_95_step is not None:
        ax.axvline(
            grok_95_step,
            color="#ECA82C",
            linestyle=":",
            linewidth=1.4,
            label=f"Test ≥95% (step {grok_95_step:,})",
        )


def pc_label(index):
    return f"PC{index + 1} ({100.0 * explained_ratio_np[index]:.2f}% var.)"


def mark_events_2d(ax, points):
    ax.scatter(*points[memorization_index], color="#B279A2", s=115, marker="*", label="Memorization", zorder=5)
    ax.scatter(*points[grok_90_index], color="#54A24B", s=90, marker="D", label="Grokking", zorder=5)
    ax.scatter(*points[-1], color="#E45756", s=90, marker="X", label="End", zorder=5)


def mark_events_3d(ax, points):
    ax.scatter(*points[memorization_index], color="#B279A2", s=115, marker="*", label="Memorization", depthshade=False)
    ax.scatter(*points[grok_90_index], color="#54A24B", s=90, marker="D", label="Grokking", depthshade=False)
    ax.scatter(*points[-1], color="#E45756", s=90, marker="X", label="End", depthshade=False)


fig, ax = plt.subplots(figsize=(10, 5.6))
ax.plot(checkpoint_steps, train_accuracies_np, color="#4C78A8", linewidth=2.0, label="Train accuracy")
ax.plot(checkpoint_steps, test_accuracies_np, color="#F58518", linewidth=2.0, label="Held-out accuracy")
ax.axhline(chance_accuracy, color="0.45", linestyle=":", linewidth=1.2, label=f"Chance ({100 / MODULUS:.2f}%)")
add_event_lines(ax)
ax.set(xlabel="Optimizer update", ylabel="Accuracy", title="Grokking on modular addition mod 23")
ax.set_ylim(-0.02, 1.03)
ax.grid(alpha=0.25)
ax.legend(loc="center right", frameon=True)
fig.tight_layout()
path_accuracy = OUT_DIR / "grokking_accuracy.png"
fig.savefig(path_accuracy, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5.6))
ax.plot(checkpoint_steps, train_losses_np, color="#4C78A8", linewidth=2.0, label="Train MSE")
ax.plot(checkpoint_steps, test_losses_np, color="#F58518", linewidth=2.0, label="Held-out MSE")
add_event_lines(ax)
ax.set_yscale("log")
ax.set(xlabel="Optimizer update", ylabel="Mean-squared error", title="Loss during memorization and delayed generalization")
ax.grid(alpha=0.25, which="both")
ax.legend(frameon=True)
fig.tight_layout()
path_losses = OUT_DIR / "grokking_losses.png"
fig.savefig(path_losses, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(checkpoint_steps, parameter_norms_np, color="#72B7B2", linewidth=2.0)
add_event_lines(ax)
ax.set(xlabel="Optimizer update", ylabel=r"$\|w\|_2$", title="Parameter norm under coupled L2 decay")
ax.grid(alpha=0.25)
ax.legend(frameon=True)
fig.tight_layout()
path_norm = OUT_DIR / "grokking_parameter_norm.png"
fig.savefig(path_norm, dpi=190, bbox_inches="tight")
plt.show()
plt.close(fig)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.plot(
    sgd_3d_np[:, 0], sgd_3d_np[:, 1], sgd_3d_np[:, 2],
    color="#4C78A8", linewidth=0.75, alpha=0.38, label="Raw minibatch SGD checkpoints",
)
ax.plot(
    avg_3d_np[:, 0], avg_3d_np[:, 1], avg_3d_np[:, 2],
    color="#F58518", linewidth=2.1, alpha=0.96, label="Exact trailing average (50 updates)",
)
mark_events_3d(ax, avg_3d_np)
ax.set_xlabel(pc_label(0), labelpad=10)
ax.set_ylabel(pc_label(1), labelpad=10)
ax.set_zlabel(pc_label(2), labelpad=10)
ax.set_title("Grokking trajectory: top-3 PCA of trailing averages")
ax.view_init(elev=25, azim=43)
ax.legend(loc="best", frameon=True)
fig.tight_layout()
path_3d = OUT_DIR / "grokking_trajectory_pca_3d.png"
fig.savefig(path_3d, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 7))
ax.plot(
    sgd_2d_np[:, 0], sgd_2d_np[:, 1],
    color="#4C78A8", linewidth=0.75, alpha=0.4, label="Raw minibatch SGD checkpoints",
)
ax.plot(
    avg_2d_np[:, 0], avg_2d_np[:, 1],
    color="#F58518", linewidth=2.1, alpha=0.96, label="Exact trailing average (50 updates)",
)
mark_events_2d(ax, avg_2d_np)
ax.set(xlabel=pc_label(0), ylabel=pc_label(1), title="Grokking trajectory: top-2 PCA variant")
ax.grid(alpha=0.25)
ax.legend(frameon=True)
fig.tight_layout()
path_2d = OUT_DIR / "grokking_trajectory_pca_2d.png"
fig.savefig(path_2d, dpi=220, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.8))
shown_components = 10
positions = np.arange(1, shown_components + 1)
ax.bar(positions, 100.0 * explained_ratio_np[:shown_components], color="#72B7B2")
ax.set_xticks(positions)
ax.set(
    xlabel="Principal component",
    ylabel="Explained variance (%)",
    title="PCA spectrum of the grokking trailing-average path",
)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
path_variance = OUT_DIR / "grokking_pca_explained_variance.png"
fig.savefig(path_variance, dpi=190, bbox_inches="tight")
plt.show()
plt.close(fig)

output_paths = [
    path_accuracy,
    path_losses,
    path_norm,
    path_3d,
    path_2d,
    path_variance,
]
for output_path in output_paths:
    print(f"Saved {output_path} ({output_path.stat().st_size / 1024:.1f} KiB)")
